In [3]:
from langchain_openai import ChatOpenAI
from langchain_classic.prompts import PromptTemplate
from dotenv import dotenv_values

config = dotenv_values(".env")
llm = ChatOpenAI(
    api_key=config["OPENROUTER_API"],
    base_url="https://openrouter.ai/api/v1",
    model="openrouter/free",
    temperature=0.8,
)

In [4]:
# ── Document Loading ──────────────────────────────────────────────
from langchain_classic.document_loaders import TextLoader
from langchain_community.document_loaders import WebBaseLoader

# 1. Load local knowledge base
text_loader = TextLoader("osama_eslam_personal_knowledge_base.txt")
text_documents = text_loader.load()

# 2. Load web pages (LinkedIn profile & GitHub repo)
web_loader = WebBaseLoader(
    web_paths=[
        "https://www.linkedin.com/in/osamaeslam",
        "https://github.com/Osama2004o/Product-Market-Search"
    ],
    requests_kwargs={"headers": {"User-Agent": "Mozilla/5.0"}}
)
web_documents = web_loader.load()

# Combine all documents
all_documents = text_documents + web_documents
print(f"Loaded {len(text_documents)} text doc(s) + {len(web_documents)} web doc(s) = {len(all_documents)} total")

# ── Chunking ──────────────────────────────────────────────────────
from langchain_classic.text_splitter import TokenTextSplitter
splitter = TokenTextSplitter(
    chunk_size = 700,
    chunk_overlap = 100
)

chunks = splitter.split_documents(all_documents)
print(f"Total chunks: {len(chunks)}")

# ── Embedding ─────────────────────────────────────────────────────
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
embedding = HuggingFaceBgeEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs = {"device":"cuda"}
)

vectors = embedding.embed_documents([i.page_content for i in chunks])

# ── Vector Store ──────────────────────────────────────────────────
from langchain_classic.vectorstores import FAISS
vectorDB = FAISS.from_documents(chunks, embedding)

USER_AGENT environment variable not set, consider setting it to identify your requests.


Loaded 1 text doc(s) + 2 web doc(s) = 3 total
Total chunks: 18


C:\Users\osama\AppData\Local\Temp\ipykernel_20992\2040511382.py:35: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceBgeEmbeddings(


In [12]:
print(web_documents)

[Document(metadata={'source': 'https://www.linkedin.com/in/osamaeslam', 'language': 'No language found.'}, page_content='\n\n\n'), Document(metadata={'source': 'https://github.com/Osama2004o/Product-Market-Search', 'title': 'GitHub - Osama2004o/Product-Market-Search · GitHub', 'description': 'Contribute to Osama2004o/Product-Market-Search development by creating an account on GitHub.', 'language': 'en'}, page_content='\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nGitHub - Osama2004o/Product-Market-Search · GitHub\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSkip to content\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nNavigation MenuSign inAppearance settingsPlatformAI CODE CREATIONGitHub CopilotWrite better cod

In [5]:
temp = """
You are a helpful assistant. Answer the question based on the context below.
The context may include information from a personal knowledge base,
a LinkedIn profile, and a GitHub repository.
If you don't know the answer,
just say "I don't know", don't try to make up an answer.
Answer in 100 words or less.

Context: {context}
Question: {question}

"""
temp = PromptTemplate.from_template(temp)

query = "what is the main field of osama eslam? and what project did he do"

similar_docs = vectorDB.similarity_search_with_score(query, k=3)

context = []
for i in similar_docs:
    context.append(i[0].page_content)

prompt = temp.format(context="\n".join(context), question=query)
print(prompt)


You are a helpful assistant. Answer the question based on the context below.
The context may include information from a personal knowledge base,
a LinkedIn profile, and a GitHub repository.
If you don't know the answer,
just say "I don't know", don't try to make up an answer.
Answer in 100 words or less.

Context:  & Orange Digital Center (July 2026).

Q: What repositories has Osama published or contributed to on GitHub?
A: Key repositories include:
- DoCrops Graduation Project: https://github.com/Osama2004o/DoCrops-FCAI-Graduation-Project
- AI Video Generation Platform: https://github.com/MoHatemTC/ai-video-generation-app
- Product Market Search Multi-Agent: https://github.com/Osama2004o/Product-Market-Search
- Mini-RAG Implementation: https://github.com/Osama2004o/mini-rag
- Personal GitHub Profile: https://github.com/Osama2004o

Q: What internships and professional experiences has Osama completed?
A: Osama's experience includes:
- SpaceTech (Sep 2026 â€“ Present): AI Engineer Inter

In [6]:
response = llm.invoke(prompt).content

In [7]:
from IPython.display import display, Markdown
display(Markdown(response))


Osama Eslam’s main field is AI Engineering, specializing in Large Language Models, Retrieval-Augmented Generation, and multi-agent systems. He graduated with a B.Sc. in Artificial Intelligence from Cairo University (2022–2026). His flagship graduation project is “DoCrops – Smart Agriculture Monitor,” an offline-capable system featuring a 98.15% accurate leaf disease classifier, a bilingual RAG advisor, and a microservice-based decision engine served via FastAPI and .NET Core to a Flutter client.

In [13]:
query = "what is all information about his product market research ?"

similar_docs = vectorDB.similarity_search_with_score(query, k=3)

context = []
for i in similar_docs:
    context.append(i[0].page_content)

prompt = temp.format(context="\n".join(context), question=query)
print(prompt)


You are a helpful assistant. Answer the question based on the context below.
The context may include information from a personal knowledge base,
a LinkedIn profile, and a GitHub repository.
If you don't know the answer,
just say "I don't know", don't try to make up an answer.
Answer in 100 words or less.

Context:  & Orange Digital Center (July 2026).

Q: What repositories has Osama published or contributed to on GitHub?
A: Key repositories include:
- DoCrops Graduation Project: https://github.com/Osama2004o/DoCrops-FCAI-Graduation-Project
- AI Video Generation Platform: https://github.com/MoHatemTC/ai-video-generation-app
- Product Market Search Multi-Agent: https://github.com/Osama2004o/Product-Market-Search
- Mini-RAG Implementation: https://github.com/Osama2004o/mini-rag
- Personal GitHub Profile: https://github.com/Osama2004o

Q: What internships and professional experiences has Osama completed?
A: Osama's experience includes:
- SpaceTech (Sep 2026 â€“ Present): AI Engineer Inter

In [14]:
response = llm.invoke(prompt).content

In [15]:
from IPython.display import display, Markdown
display(Markdown(response))

Osama’s product‑market‑research project is the **Product Market Search Multi‑Agent** system (GitHub: https://github.com/Osama2004o/Product-Market-Search). It autonomously scrapes live product data from Amazon Egypt, Noon Egypt, and Jumia Egypt using Playwright/Browser automation, normalizes attributes and currencies, and ranks items via a price‑to‑value score. The multi‑agent orchestrator (CrewAI) includes dedicated scraper agents, a NormalizerAgent, and a RankingAgent. Backend is FastAPI (async REST), frontend is a React/Vite glassmorphism UI, and the stack also uses Google Gemini, LangChain, Beautiful Soup, and Docker‑compatible services. The platform delivers real‑time deal comparison, interactive filters, and direct product links.